# M3 H1: hyperparameter tuning of the winning configuration

We tune four hyperparameters of the FE winner (`BASE_CONFIG`, a run id in `configs/m3/`) to answer H1: which settings
suit the chosen model, and which of them matter. The four are learning rate and weight decay on log scales,
augmentation strength, and one margin: the ArcFace margin for an ArcFace head, else the triplet margin for a triplet
loss, else the embedding size for plain cross-entropy. Each trial trains `EPOCHS` epochs and validates every
`EVAL_EVERY`.

**Inputs:** `serpentguard-indian-snakes-clean`, `serpentguard-sam-masks`, `serpentguard-m3-subset` and, for a
front-end winner, `serpentguard-m3-frontend-preds` (optional: without it `PREDS` is None), found with
`kagglehub.dataset_download`. The code comes from `../src` and `../configs/m3`, which the runner kernel unpacks into
`/kaggle/working/repo`.

**Outputs** in `/kaggle/working/`: `H1_s<SAMPLER_SEED>.db`, `H1_s<SAMPLER_SEED>_trials.csv` and
`H1_s<SAMPLER_SEED>/`, which holds one run folder per trial (`trial_000/`, ...) and each worker's log. A merge adds
`H1_merged.db`, the plots in `plots/` (PNG at 150 dpi) and the retrain configs in `configs/m3/`.

**Run:** on Kaggle with two GPUs and internet on (pip and the ImageNet weights). `BASE_CONFIG` must be set at launch;
the notebook stops without it rather than tune the wrong model. `SAMPLER_SEED` picks the slice, and `SLICE_DBS`,
`DEFAULT_RUNS` and `TUNED_RUNS` turn on the merge and the plots after the retrain.

## How the search runs

**One kernel runs one slice.** It runs `N_TRIALS` trials of its own study, drawn by `SAMPLER` with `SAMPLER_SEED`, in
two worker processes, one per GPU, that share the kernel's own SQLite file. We launch about six kernels of four trials
each, seeds 1 to 6, with the random sampler. The seed names the slice (`H1_s<SAMPLER_SEED>`), so two kernels share a
slice, and draw the same points, only if they are given the same seed; a merge refuses repeated points.

**Pruning is off at launch** (`PRUNE = False`), so every trial runs its `EPOCHS` and the merged slices are a clean
random sample for the parameter importances. With `PRUNE = True`, a median pruner stops a trial whose best validation
macro-F1 so far falls below the median of the completed trials at the same epoch, checked from the first evaluation on
but never at the last. It sees only its own slice's trials, which makes it weaker than one pruner over the whole
search, but still sound: it only compares trials of the same setup at the same epoch. A four-trial slice then waits
for two completed trials (optuna's default of five would never prune a slice that small).

**Failed trials and checkpoints.** A trial that fails (out of GPU memory, say) is recorded as failed, and its worker
goes on to its next trial; running the search cell again replaces it. Each trial drops its `best.pt`, prototypes and
saved outputs when it ends (`KEEP_CHECKPOINTS = False`) and keeps `config.yaml`, `versions.json`, `subset_hash.txt`,
`metrics.jsonl` and, when it completes, `final_metrics.json`. We read the search only for its scores; the retrain
remakes the top configs' checkpoints.

**Merge and retrain.** When `SLICE_DBS` is set (with `N_TRIALS = 0` if no search should run), the merge puts every
slice's trials into one study, `/kaggle/working/H1_merged.db`, reading each slice file from a temporary copy, never
from the read-only input. The optimisation-history, parallel-coordinates and parameter-importance plots come from the
merged study, and so do `configs/m3/H1_top{1,2,3}_s{1,2}.yaml`: the three best trials as full-length retrain configs,
which notebook 12 runs unchanged once they are committed. After the retrain, `DEFAULT_RUNS` and `TUNED_RUNS` add the
macro-F1-by-epoch and by-tier plots. N1 is the final configuration at seeds 42, 1 and 2, and its configs come from the
winning file: `tune.n1_configs("configs/m3/H1_top<k>_s1.yaml")` writes only `N1_s42.yaml`, since the retrain already
ran seeds 1 and 2; given the default recipe's YAML, whose own run is at seed 42, it writes `N1_s1.yaml` and
`N1_s2.yaml`.

In [1]:
BASE_CONFIG = 'R_F0'
SAMPLER = 'random'
SAMPLER_SEED = 1
N_TRIALS = 4
EPOCHS = 8
EVAL_EVERY = 2
PRUNE = False
KEEP_CHECKPOINTS = False
SLICE_DBS = []            # merge: the slices' H1_s<seed>.db files (this kernel's own is added when it searched)
DEFAULT_RUNS = []         # after the retrain: the default recipe's run folders, for the last two plots
TUNED_RUNS = []           # after the retrain: the tuned configuration's run folders (one per seed)

In [2]:
import os
import subprocess
import sys

ALBUMENTATIONS_VERSION = "2.0.8"  # requirements-m3.txt; the per-transform augmentation seeding was verified on it
TIMM_VERSION = "1.0.31"            # requirements-m3.txt; the encoders' feature maps were measured on it
OPTUNA_VERSION = "5.0.0"           # requirements-m3.txt; the study, sampler, pruner and storage were tested on it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"albumentations=={ALBUMENTATIONS_VERSION}",
                f"timm=={TIMM_VERSION}", f"optuna=={OPTUNA_VERSION}"], check=True)
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"  # otherwise albumentations asks PyPI for its latest version on import

import albumentations
import optuna
import timm

assert albumentations.__version__ == ALBUMENTATIONS_VERSION, f"albumentations {albumentations.__version__} is loaded"
assert timm.__version__ == TIMM_VERSION, f"timm {timm.__version__} is loaded"
assert optuna.__version__ == OPTUNA_VERSION, f"optuna {optuna.__version__} is loaded"
print(f"albumentations {albumentations.__version__}, timm {timm.__version__}, optuna {optuna.__version__}")

albumentations 2.0.8, timm 1.0.31, optuna 5.0.0


In [3]:
SCIKIT_LEARN_VERSION = "1.9.1"  # requirements-m3.txt; the runner's metrics and fANOVA use it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={SCIKIT_LEARN_VERSION}"], check=True)

import sklearn

assert sklearn.__version__ == SCIKIT_LEARN_VERSION, (f"scikit-learn {sklearn.__version__} is loaded; restart the "
                                                     f"kernel after the install")
print(f"scikit-learn {sklearn.__version__}")

scikit-learn 1.9.1


In [4]:
import logging
from pathlib import Path

import kagglehub

CODE = Path("/kaggle/input/serpentguard-code/src")
CODE = CODE if CODE.exists() else Path("../src").resolve()
sys.path.insert(0, str(CODE))
CONFIGS = CODE.parent / "configs" / "m3"

import pandas as pd
import torch
from IPython.display import Image, display

from serpentguard import tune
from serpentguard.config import load_config

logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)
OWNER = "puneetbajajiitm"


def input_root(slug):
    """Where an attached dataset is mounted; Kaggle no longer mounts datasets at /kaggle/input/<slug>."""
    return Path(kagglehub.dataset_download(f"{OWNER}/{slug}"))


DATA = input_root("serpentguard-indian-snakes-clean")
MASKS = input_root("serpentguard-sam-masks")
SUBSET = input_root("serpentguard-m3-subset") / "subset.csv"
try:
    PREDS = input_root("serpentguard-m3-frontend-preds")  # only front-end winners read it
except Exception as error:  # neither attached nor downloadable
    print(f"serpentguard-m3-frontend-preds unavailable ({type(error).__name__}: {error})")
    PREDS = None
OUT = Path("/kaggle/working")
STUDY_NAME = f"H1_s{SAMPLER_SEED}"  # the seed names the slice, so two kernels never share one by accident
STUDY_FILE = OUT / f"{STUDY_NAME}.db"
STORAGE = f"sqlite:///{STUDY_FILE}"
SLICE_DIR = OUT / STUDY_NAME
MERGED_FILE = OUT / "H1_merged.db"
PLOTS = OUT / "plots"
RETRAIN_CONFIGS = OUT / "configs" / "m3"
EXPECTED_SUBSET_HASH = "40196d19a68e"  # first 12 hex digits of the real subset's SHA-256
# Optuna's default waits for five completed trials, which a four-trial slice never has; half the slice lets its later
# trials be pruned against the first ones.
PRUNER_STARTUP = min(tune.PRUNER_STARTUP_TRIALS, max(1, N_TRIALS // 2))

assert BASE_CONFIG, "set BASE_CONFIG to the FE winner's run id in configs/m3/"
assert SAMPLER in tune.SAMPLERS, f"SAMPLER must be one of {tune.SAMPLERS}"
for required in (DATA / "metadata.csv", DATA / "classes.csv", DATA / "data", MASKS / "mask_stats.csv", SUBSET,
                 CONFIGS / f"{BASE_CONFIG}.yaml"):
    assert required.exists(), f"missing input {required}"
base = load_config(CONFIGS / f"{BASE_CONFIG}.yaml")
gpus = torch.cuda.device_count()
assert gpus >= 1 or not N_TRIALS, "no GPU in this session: the search would train on the CPU"
workers = max(1, (os.cpu_count() or 2) // max(1, gpus))
pruning = f"pruning after {PRUNER_STARTUP} completed" if PRUNE else "pruning off"
print(f"code {CODE}; {gpus} GPUs, {os.cpu_count()} CPUs, {workers} loader workers per trial; slice {STUDY_NAME}: "
      f"{N_TRIALS} trials, {SAMPLER} sampler, seed {SAMPLER_SEED}, {pruning}, checkpoints "
      f"{'kept' if KEEP_CHECKPOINTS else 'dropped'}; base "
      f"{base.run_id} ({base.head} head, losses {base.losses}): tuning {', '.join(tune.search_space(base))}, kept at "
      f"the base value {tune.fixed_params(base) or 'nothing'}; front-end predictions {PREDS or 'not attached'}")

serpentguard-m3-frontend-preds unavailable (BackendError: POST failed with: {"errors":["Not found"],"error":{"code":5},"wasSuccessful":false})
code /kaggle/working/repo/src; 2 GPUs, 4 CPUs, 2 loader workers per trial; slice H1_s1: 4 trials, random sampler, seed 1, pruning off, checkpoints dropped; base R_F0 (arcface head, losses ['arcface']): tuning lr, weight_decay, aug_strength, arcface_m, kept at the base value nothing; front-end predictions not attached


In [5]:
from serpentguard.run_data import WARM_THREADS, run_files, warm_page_cache

# Every trial opens the same files (the search changes no input); reading them once, 64 at a time, puts them in the
# page cache that both worker processes share.
if N_TRIALS:
    warmed = warm_page_cache(sorted(run_files(base, DATA, MASKS, SUBSET, PREDS)), threads=WARM_THREADS)
    print(f"page cache: {warmed['files']:,} files ({warmed['missing']} missing), {warmed['mb']:,.0f} MB in "
          f"{warmed['seconds']:.0f} s")

page cache: 39,212 files (0 missing), 5,386 MB in 301 s


In [6]:
from collections import Counter

study = tune.run_study(base, n_trials=N_TRIALS, epochs=EPOCHS, storage=STORAGE, data_root=DATA, masks_root=MASKS,
                       subset_csv=SUBSET, out_dir=SLICE_DIR, preds_root=PREDS, eval_every=EVAL_EVERY,
                       num_workers=workers, study_name=STUDY_NAME, startup_trials=PRUNER_STARTUP, sampler=SAMPLER,
                       sampler_seed=SAMPLER_SEED, prune=PRUNE, keep_checkpoints=KEEP_CHECKPOINTS)
print(tune.progress(STUDY_NAME, STORAGE, N_TRIALS))
# A failed trial doesn't stop its worker, so a systematic error (a wrong path, say) fails every trial while the workers
# exit cleanly. We fail the kernel then, rather than let that pass for a search.
if N_TRIALS:  # a merge-only kernel searches nothing
    states = Counter(trial.state.name for trial in study.trials)
    print("trials by state: "
          + ", ".join(f"{state.lower()} {states[state]}" for state in ("COMPLETE", "PRUNED", "FAIL")))
    failed = [trial for trial in study.trials if trial.state.name == "FAIL"]
    if failed:
        first = failed[0]
        log = SLICE_DIR / f"worker_{first.user_attrs.get('worker')}.log"
        print(f"{len(failed)} trials failed; the first, trial {first.number}: "
              f"{first.user_attrs.get('error', 'no exception recorded')} (full traceback in {log})")
    assert states["COMPLETE"], (f"no trial of {STUDY_NAME} completed ({dict(states)}): a systematic error, not a "
                                f"search; see the failure above and the worker logs in {SLICE_DIR}")

[I 2026-11-04 23:52:07,843] A new study created in RDB with name: H1_s1


worker 0 (cuda:0, CUDA_VISIBLE_DEVICES=0): 2 trials, log /kaggle/working/H1_s1/worker_0.log
worker 1 (cuda:0, CUDA_VISIBLE_DEVICES=1): 2 trials, log /kaggle/working/H1_s1/worker_1.log
5.0 min: 0 complete, 0 pruned, 2 running, 0 failed of 4
10.0 min: 0 complete, 0 pruned, 2 running, 0 failed of 4
15.0 min: 0 complete, 0 pruned, 2 running, 0 failed of 4
20.0 min: 0 complete, 0 pruned, 2 running, 0 failed of 4
25.0 min: 0 complete, 0 pruned, 2 running, 0 failed of 4
30.0 min: 0 complete, 0 pruned, 2 running, 0 failed of 4
35.0 min: 0 complete, 0 pruned, 2 running, 0 failed of 4
40.0 min: 0 complete, 0 pruned, 2 running, 0 failed of 4
45.0 min: 2 complete, 0 pruned, 2 running, 0 failed of 4; best macro-F1 0.2054 (trial 0)
50.0 min: 2 complete, 0 pruned, 2 running, 0 failed of 4; best macro-F1 0.2054 (trial 0)
55.0 min: 2 complete, 0 pruned, 2 running, 0 failed of 4; best macro-F1 0.2054 (trial 0)
60.0 min: 2 complete, 0 pruned, 2 running, 0 failed of 4; best macro-F1 0.2054 (trial 0)
65.0 

In [7]:
rows = []
for trial in study.trials:
    finished = trial.datetime_complete is not None and trial.datetime_start is not None
    rows.append({"trial": trial.number, "state": trial.state.name.lower(), "best macro-F1": trial.value,
                 "last epoch": max(trial.intermediate_values, default=None),
                 "best epoch": trial.user_attrs.get("best_epoch"), "worker": trial.user_attrs.get("worker"),
                 **trial.params,
                 "minutes": (trial.datetime_complete - trial.datetime_start).total_seconds() / 60 if finished else None})
    if trial.state in tune.FINISHED:  # completed and pruned trials alike wrote the hash of the subset they read
        digest = (Path(trial.user_attrs["run_dir"]) / "subset_hash.txt").read_text().strip()
        assert digest.startswith(EXPECTED_SUBSET_HASH), f"trial {trial.number} read another subset ({digest})"
trials = pd.DataFrame(rows)
trials.to_csv(OUT / f"{STUDY_NAME}_trials.csv", index=False)
trials.sort_values("best macro-F1", ascending=False).round(5) if rows else trials

,trial,state,best macro-F1,last epoch,best epoch,worker,lr,weight_decay,aug_strength,arcface_m,minutes
0,0,complete,0.20540,8,8,1,0.00044,0.00830,1.31690,0.25889,41.03750
1,1,complete,0.19438,8,8,0,0.00020,0.00360,0.59854,0.22876,40.86556
2,2,complete,0.18712,8,8,0,0.00017,0.02449,0.80994,0.10519,40.79836
3,3,complete,0.14533,8,8,1,0.00015,0.00028,1.49308,0.37157,40.82565


## Merge

Every slice in `SLICE_DBS` (and this kernel's own, when it searched) goes into one study; the plots and the top-3 retrain configs come from it. Nothing runs while `SLICE_DBS` is empty.

In [8]:
if SLICE_DBS:
    slice_dbs = list(dict.fromkeys(Path(path).resolve() for path in [*SLICE_DBS, *([STUDY_FILE] if N_TRIALS else [])]))
    MERGED_FILE.unlink(missing_ok=True)  # derived: rebuilt from the slices every time
    merged = tune.merge_studies(slice_dbs, tune.STUDY_NAME, storage=f"sqlite:///{MERGED_FILE}")
    print(f"merged {len(merged.trials)} finished trials from {len(merged.user_attrs['slices'])} slices into "
          f"{MERGED_FILE}: {merged.user_attrs['slices']}")
    for path in tune.save_study_plots(merged, PLOTS).values():
        display(Image(filename=str(path)))

In [9]:
if SLICE_DBS:
    for path in tune.top_configs(merged, k=3, out_dir=RETRAIN_CONFIGS):
        print(f"--- {path}")
        print(path.read_text())

## After the retrain

Macro-F1 by epoch and by tier, from `DEFAULT_RUNS` and `TUNED_RUNS`; nothing runs while they are empty.

In [10]:
if DEFAULT_RUNS and TUNED_RUNS:
    for path in tune.save_run_plots([Path(run) for run in DEFAULT_RUNS], [Path(run) for run in TUNED_RUNS],
                                    PLOTS).values():
        display(Image(filename=str(path)))